# Example 8.4.1 / 1: Read Status, Reduced Costs and Duals

Companion notebook for **8.4.1 Solver Status and Sensitivity Suffixes** in the AMPL lab Documentation, section "Read Status, Reduced Costs and Duals". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Read Status, Reduced Costs and Duals

`.sstatus` gives each variable's and constraint's status in the solver's basis, `.rc` a
variable's reduced cost, and `.dual` a constraint's dual value. This example solves the steel
model with HiGHS and reads all three. In `data.py` and `run.py`, `ampl` is an `AMPL` object that
has already read `model.mod`; `ampl.get_data(...)` returns several suffixes as one table:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# production hours fit the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.get_data("Make, Make.sstatus, Make.rc").to_pandas())
print("Time:", ampl.get_value("Time.sstatus"), ampl.get_value("Time.dual"))

       Make Make.sstatus   Make.rc
bands  6000          upp  1.800000
coils     0          low -3.142857
plate  1600          bas  0.000000
Time: upp 4640


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display Make, Make.sstatus, Make.rc;
display Time.sstatus, Time.dual;

:       Make Make.sstatus   Make.rc     :=
bands   6000   upp           1.8
coils      0   low          -3.14286
plate   1600   bas           0
;

Time.sstatus = upp
Time.dual = 4640



### Another way in amplpy

`plate` is basic (`bas`) and its reduced cost is `0`. `bands` sits at its upper bound (`upp`):
its reduced cost `1.8` is the profit gained per extra ton of market. `coils` sits at its lower
bound (`low`): each ton forced into the plan would cost `3.14286`. `Time` is at its upper
limit (`upp`, the status of its slack variable), and its dual `4640` is the profit of one more
hour.

Gurobi's `alg:sens=1` adds sensitivity ranges. With the same `model.mod` and `data.py`, the
variables get `.down` and `.up`, the range of their profit coefficient (`.current`) over which
this basis stays optimal, and the one-sided constraint `Time` gets the range of its right-hand
side. `ampl.display(...)` prints the table in AMPL's format, where an unlimited range shows as
`1e+100`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [11]:
# run.py
ampl.solve(solver="gurobi", gurobi_options="alg:sens=1", verbose=False)
ampl.display("Make.down", "Make.current", "Make.up")
print("Time rhs range:", ampl.get_value("Time.down"), ampl.get_value("Time.up"))

:       Make.down  Make.current     Make.up      :=
bands        23.2        25       1e+100
coils   -1e+100          30           33.1429
plate        26.25       29           31.25
;

Time rhs range: 30 51.875


### Pure AMPL: data.dat and model.run

The same in pure AMPL:

In [12]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Overwriting data.dat


In [13]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [14]:
%%ampl_eval
# model.run
option solver gurobi;
option gurobi_options 'alg:sens=1';
option solver_msg 0;
solve;
display Make.down, Make.current, Make.up;
display Time.down, Time.up;

Gurobi 13.0.0:   alg:sens = 1

suffix up OUT;
suffix down OUT;
suffix current OUT;
suffix sensobj OUT;
suffix senslbhi OUT;
suffix senslblo OUT;
suffix sensubhi OUT;
suffix sensublo OUT;
suffix sensobjhi OUT;
suffix sensobjlo OUT;
suffix sensrhshi OUT;
suffix sensrhslo OUT;
:       Make.down  Make.current     Make.up      :=
bands        23.2        25       1e+100
coils   -1e+100          30           33.1429
plate        26.25       29           31.25
;

Time.down = 30
Time.up = 51.875

